# smmap: 5 Basic Usage Examples

In [ ]:
# Example: Basic smmap usage for memory-mapped file access
import tempfile
import os
from smmap import SlidingWindowMapManager

# Create a temporary file with some data
with tempfile.NamedTemporaryFile(mode='w+b', delete=False) as temp_file:
    test_data = b"Hello, this is a test file for smmap memory mapping!\nLine 2\nLine 3"
    temp_file.write(test_data)
    temp_filename = temp_file.name

try:
    # Create a sliding window map manager
    with open(temp_filename, 'rb') as f:
        manager = SlidingWindowMapManager()
        
        # Map a region of the file
        mapped_region = manager.make_cursor(f.fileno()).use_region(0, len(test_data))
        
        # Read data from the memory-mapped region
        print("Region size:", mapped_region.size())
        
        # Convert memoryview to bytes for operations
        buffer_data = bytes(mapped_region.buffer())
        print("First 20 bytes:", buffer_data[:20])
        print("Full content:", buffer_data.decode('utf-8'))
        
        # Check if we can access the data
        print("Contains 'smmap':", b'smmap' in buffer_data)
        print("Number of lines:", buffer_data.count(b'\n'))
        
finally:
    # Clean up the temporary file
    os.unlink(temp_filename)
    print("Temporary file cleaned up")

In [ ]:
# Example 2: Create a memory map manager
# smmap 5 exports the managers at the top level (`smmap.mman` is the module that defines them; there is no `Mman`)
from smmap import StaticWindowMapManager

mman = StaticWindowMapManager()
print(f'Memory map manager created: {mman}')
assert (mman.num_file_handles(), mman.mapped_memory_size(), mman.window_size()) == (0, 0, 0)
print(f'Max mapped memory: {mman.max_mapped_memory_size()} bytes')

In [ ]:
# Example 3: Map a file using correct smmap API
import os
from smmap import SlidingWindowMapManager

# Create a test file
with open('testfile.txt', 'w') as f:
    f.write('hello world from smmap test!')

try:
    # Use SlidingWindowMapManager with proper file handling
    manager = SlidingWindowMapManager()
    
    with open('testfile.txt', 'rb') as f:
        # Create cursor and map a region
        cursor = manager.make_cursor(f.fileno())
        region = cursor.use_region(0, 11)  # Map first 11 bytes "hello world"
        
        # Read from the mapped region
        buffer_data = bytes(region.buffer())
        print("Mapped data:", buffer_data)
        print("As string:", buffer_data.decode('utf-8'))
        print("Region size:", region.size())
        
        # Map a different region
        region2 = cursor.use_region(6, 5)  # Map "world" (5 bytes starting at position 6)
        buffer_data2 = bytes(region2.buffer())
        print("Second region:", buffer_data2.decode('utf-8'))

finally:
    # Clean up test file
    if os.path.exists('testfile.txt'):
        os.remove('testfile.txt')
        print("Test file cleaned up")

In [ ]:
# Example: Basic smmap buffer management and window sliding
import tempfile
import os
from smmap import SlidingWindowMapManager

# Create a larger test file to demonstrate sliding windows
test_content = "Line 1: This is the first line of our test file.\n" \
               "Line 2: Here's some more content for testing.\n" \
               "Line 3: Even more data to work with sliding windows.\n" \
               "Line 4: Final line with additional test data."

with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as f:
    f.write(test_content)
    temp_file = f.name

try:
    manager = SlidingWindowMapManager(window_size=50, max_memory_size=200)
    
    with open(temp_file, 'rb') as f:
        cursor = manager.make_cursor(f.fileno())
        
        # Read first 30 bytes
        region1 = cursor.use_region(0, 30)
        print("First 30 bytes:", bytes(region1.buffer()).decode('utf-8'))
        
        # Slide to read bytes 25-55 (overlapping)
        region2 = cursor.use_region(25, 30)
        print("Bytes 25-55:", bytes(region2.buffer()).decode('utf-8'))
        
        # Read from end of file
        file_size = len(test_content.encode('utf-8'))
        region3 = cursor.use_region(file_size - 30, 30)
        print("Last 30 bytes:", bytes(region3.buffer()).decode('utf-8'))

finally:
    os.unlink(temp_file)

In [ ]:
# Example 5: Show SlidingWindowMapManager attributes and methods
from smmap import SlidingWindowMapManager
import tempfile

# Create a SlidingWindowMapManager instance
manager = SlidingWindowMapManager()

print("SlidingWindowMapManager attributes:")
attributes = [attr for attr in dir(manager) if not attr.startswith('_')]
for attr in sorted(attributes):
    print(f"  {attr}")

print(f"\nManager window size: {manager.window_size}")
if hasattr(manager, 'max_memory_size'):
    print(f"Manager max memory size: {manager.max_memory_size}")
else:
    print("Manager max memory size: Not available")

# Show some actual attribute values if they exist
if hasattr(manager, 'max_file_handles'):
    print(f"Max file handles: {manager.max_file_handles}")
if hasattr(manager, '_fdict'):
    print(f"File dictionary size: {len(manager._fdict)}")

# Create a test file and cursor to show cursor attributes
with tempfile.NamedTemporaryFile(mode='w+b') as f:
    f.write(b"Test data for cursor exploration")
    f.flush()
    
    cursor = manager.make_cursor(f.fileno())
    print(f"\nCursor type: {type(cursor).__name__}")
    print("Cursor attributes:")
    cursor_attrs = [attr for attr in dir(cursor) if not attr.startswith('_')]
    for attr in sorted(cursor_attrs):
        print(f"  {attr}")
    
    # Create a region to show region attributes
    region = cursor.use_region(0, 10)
    print(f"\nRegion type: {type(region).__name__}")
    print("Region attributes:")
    region_attrs = [attr for attr in dir(region) if not attr.startswith('_')]
    for attr in sorted(region_attrs):
        print(f"  {attr}")